[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/feature/phase-0-1-scaffolding-and-bakeoff/examples/colab_bakeoff.ipynb)

# Bake-off synthesis: side-by-side comparison of three candidates

Reads outputs from `bakeoff_{roboflow,atfa,tz}.ipynb` and produces:
- Side-by-side annotated-video panel (stratified sample of frames)
- Per-candidate quantitative summaries (detection rate, track IDs, team consistency)
- A scoring template per the spec §4.4 rubric

Run locally (not Colab): all three candidate outputs must be on disk under
`data/bakeoff_outputs/`.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("../data/bakeoff_outputs")
candidates = {
    "roboflow": pd.read_parquet(ROOT / "roboflow" / "trajectories.parquet"),
    "atfa":     pd.read_parquet(ROOT / "atfa" / "trajectories.parquet"),
    "tz":       pd.read_parquet(ROOT / "tz" / "trajectories.parquet"),
}
for name, df in candidates.items():
    print(f"{name}: {len(df)} rows, {df['frame'].nunique()} frames, {df['track_id'].nunique()} unique tracks")

In [ ]:
def summarize(df: pd.DataFrame) -> dict:
    return {
        "n_frames": int(df["frame"].nunique()),
        "n_rows": int(len(df)),
        "unique_tracks": int(df["track_id"].nunique()),
        "mean_detections_per_frame": float(df.groupby("frame").size().mean()),
        "ball_detection_rate": float((df["class"] == "ball").groupby(df["frame"]).any().mean()),
        "mean_conf": float(df["conf"].mean()),
        "team_consistency_per_track": float(
            df.groupby("track_id")["team"].apply(lambda s: s.value_counts(normalize=True).max()).mean()
        ),
    }

import json
for name, df in candidates.items():
    print(f"\n{name}:")
    print(json.dumps(summarize(df), indent=2))

In [ ]:
import cv2
import numpy as np

def grab_frame(video_path: Path, frame_idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    cap.release()
    return frame if ok else np.zeros((720, 1280, 3), dtype=np.uint8)

def side_by_side(frame_idx: int) -> np.ndarray:
    frames = [grab_frame(ROOT / name / "annotated.mp4", frame_idx) for name in ["roboflow", "atfa", "tz"]]
    target_h = min(f.shape[0] for f in frames)
    resized = [cv2.resize(f, (int(f.shape[1] * target_h / f.shape[0]), target_h)) for f in frames]
    return np.concatenate(resized, axis=1)

# Sample 8 stratified frames across the clip for visual review
import matplotlib.pyplot as plt
ref_video = cv2.VideoCapture(str(ROOT / "roboflow" / "annotated.mp4"))
total_frames = int(ref_video.get(cv2.CAP_PROP_FRAME_COUNT))
ref_video.release()
sample_idxs = np.linspace(0, total_frames - 1, 8).astype(int)
fig, axes = plt.subplots(8, 1, figsize=(18, 24))
for ax, idx in zip(axes, sample_idxs):
    composite = side_by_side(idx)
    ax.imshow(cv2.cvtColor(composite, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx} — left: roboflow, mid: atfa, right: tz")
    ax.axis("off")
plt.tight_layout()
plt.savefig("../data/bakeoff_outputs/side_by_side.png", dpi=100)
plt.show()

In [ ]:
# Manual scoring; record values 1-5 per axis after visual review
scores = {
    "roboflow": {"player_recall": None, "ball_recall": None, "track_stability": None,
                 "team_classification": None, "homography_fidelity": None, "nine_v_nine_handling": None},
    "atfa":     {"player_recall": None, "ball_recall": None, "track_stability": None,
                 "team_classification": None, "homography_fidelity": None, "nine_v_nine_handling": None},
    "tz":       {"player_recall": None, "ball_recall": None, "track_stability": None,
                 "team_classification": None, "homography_fidelity": None, "nine_v_nine_handling": None},
}
# Fill in after watching annotated.mp4 for each and inspecting side_by_side.png.
# Then run the cell below to write scores.csv:
score_df = pd.DataFrame(scores).T
score_df["total"] = score_df.sum(axis=1)
print(score_df)
score_df.to_csv("../data/bakeoff_outputs/scores.csv")